<a href="https://colab.research.google.com/github/mariobet23440/Proyecto-3---M-todos-Num-ricos-1/blob/main/Proyecto3MetodosNumericos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***MOTOR DC - GEMELO DIGITAL Y CONTROL PID***

```Proyecto 3 - Métodos Numéricos 1```

***Creado por:***
*   Mario Betancourt (23440)
*   Pongan sus nombres y (carnets)

## ***HISTORIAL DE REVISIONES***
*   21/11/2026 - Creación de archivo y método de Euler
*   22/11/2026 - Correcciones al desarrollo matemático de la Parte 1, revisadas y aprobadas por el equipo (ver `UsoIA/`)



# ***INTRODUCCIÓN***
Le damos la bienvenida al cuaderno de Jupyter del Proyecto 3 del curso Métodos Numéricos 1 de la Universidad del Valle de Guatemala. En este proyecto trabajaremos en la aplicación de métodos numéricos para calcular derivadas e integrales y métodos de resolución de ecuaciones diferenciales al análisis de la dinámica de un motor DC.

De acuerdo con la guía, el motor DC se modela a través de un sistema de tres ecuaciones diferenciales acopladas:

$$\begin{cases}
L\frac{di}{dt} = V-Ri-K_e \omega \\
J_m\frac{d\omega}{dt} = K_t i - b\omega -\tau_c \text{ sign}(\omega) - \tau_{carga} \\
\frac{d\theta}{dt} = \omega
\end{cases}$$

Donde los símbolos representan las siguientes cantidades físicas

*   $L$ - Inductancia del motor [henrios, H]
*   $V$ - Voltaje aplicado en las terminales del motor [Volts, V]
*   $R$ - Resistencia del motor [Ohms, $\Omega$]
*   $K_e$ - Constante de fuerza contraelectromotriz [$V/(rad/s)$]
*   $\omega$ - Velocidad angular del motor [radianes por segundo, $rad/s$]
*   $J_m$ - Inercia del eje del motor [$\text{kg}\cdot\text{m}^2$]
*   $K_t$ - Constante de torque del motor [$\text{N}\cdot\text{m}/\text{A}$]
*   $b$ - Coeficiente de amortiguamiento viscoso [$\text{N}\cdot\text{m}\cdot\text{s}/\text{rad}$]
*   $\tau_c$ - Torque de fricción de Coulomb [$\text{N}\cdot\text{m}$]
*   $\tau_{carga}$ - Torque de carga [$\text{N}\cdot\text{m}$]

Un estudio de estos modelos en cursos de sistemas de control sugiere que el sistema de ecuaciones representa a un motor DC controlado por armadura. Este modelo de motores DC supone que la corriente en el estator (corriente de campo) es constante y el control de movimiento se realiza por la corriente del devanado del rotor (devanado de armadura). Este modelo de motores es aplicable también a motores DC de imanes permanentes. El motor DC JGA25-370 entra dentro de esta categoría. Para más información, véase Nise

(AGREGAR REFERENCIA POR FAVOR, capítulo 4 no recuerdo donde estaba)


A diferencia del modelo lineal presentado en Nise (no sé, no sé, no sé) el modelo analizado en este proyecto agrega una dificultad adicional: *se considera un término no lineal para la fricción de Coulomb*. Aunque es posible linealizar el sistema y analizarlo a través de Transformadas de Laplace (como se hace en Sistemas de Control 1), ese no es el objetivo de este proyecto. Al fin y al cabo, el objetivo de este curso no es simplificar el modelo, sino resolverlo numéricamente tal cual. Por lo tanto, a diferencia de lo que se haría en un análisis de sistemas lineales, en este proyecto resolveremos la ecuación original con todas sus no linealidades.

# ***PARTE 1 - SOLUCIÓN DE SISTEMAS DE ECUACIONES DIFERENCIALES***

Ahora que fijamos como objetivo resolver el sistema de ecuaciones diferenciales mostrado anteriormente, surge la importante pregunta de cómo debemos proceder. Empecemos tratando de entender cómo resolver un sistema de ecuaciones a través del método de Euler.

## **1.1. El Método de Euler para una EDO de primer orden**
El método de Euler es fácil y muy intuitivo de definir para un Problema de Valores Iniciales (PVI) con una EDO de primer orden:

$$\begin{cases}
\frac{dy}{dx}=f(x,y)\\
y(x_0) = y_0
\end{cases}
$$

Donde $x_0$ y $y_0$ son constantes arbitrarias. Para aproximar la solución a este PVI, utilizamos diferencias finitas hacia adelante para aproximar la derivada en $x_k$ usando el intervalo $[x_k, x_{k+1}]$, donde $x_{k+1} = x_k + h$:

$$\frac{dy}{dx} \approx \frac{y_{k+1} - y_k}{h} $$

Donde $y_{k+1} \approx y(x_{k+1})$ y $y_{k} \approx y(x_{k})$ son las **aproximaciones numéricas** de la variable dependiente en los extremos del intervalo antes definido. (En la sección 1.4 se usará una letra mayúscula para distinguir la solución exacta.)

Al sustituir en la EDO, evaluando $f$ en el extremo izquierdo del intervalo, $(x_k, y_k)$:
$$\frac{y_{k+1} - y_k}{h}\approx f(x_k,y_k)$$

Evaluar $f$ en el extremo izquierdo es justamente lo que hace que el método sea **explícito**; evaluarla en $(x_{k+1}, y_{k+1})$ daría el método de Euler implícito.

Resuelva para $y_{k+1}$ y obtendrá la ecuación del método de Euler explícito (el método de Euler que aproxima la solución extrapolando hacia adelante usando la derivada en el punto actual):

$$y_{k+1} = y_k + hf(x_k, y_k)$$

¿Ve lo simple que es llegar a este famoso método?



## **1.2. El Método de Euler para un sistema de EDOs de primer orden**
Ahora extrapolemos esta deducción hacia un sistema de varias EDOs de variables dependientes en el vector $\mathbf{y}=[y_1, y_2, ..., y_N]^T$. ¿Qué tanto podría cambiar?

Suponga que todas las ecuaciones del sistema pueden escribirse en forma canónica (aunque el lado derecho sea una no linealidad espeluznante):

$$
\begin{cases}
y_1' = f_1(x,y_1,y_2, ..., y_N)\\
y_2' = f_2(x,y_1,y_2, ..., y_N)\\
y_3' = f_3(x,y_1,y_2, ..., y_N)\\
\vdots \\
y_N' = f_N(x,y_1,y_2, ..., y_N)
\end{cases}
$$

Aunque no lo crea, este sistema de ecuaciones grita que puede ser expresado en forma vectorial. Reúna las variables dependientes en el vector $\mathbf{y}$ y los lados derechos en una sola función vectorial $\mathbf{f}$, el **campo vectorial** del sistema:

$$
\mathbf{y} = \begin{pmatrix} y_1 \\ y_2 \\ \vdots \\ y_N \end{pmatrix},\qquad
\mathbf{f}(x,\mathbf{y}) = \begin{pmatrix} f_1(x,\mathbf{y}) \\ f_2(x,\mathbf{y}) \\ \vdots \\ f_N(x,\mathbf{y}) \end{pmatrix}
$$

Si recuerda que derivar una función vectorial equivale a derivar cada una de sus componentes, $\mathbf{y}' = [y_1', y_2', ..., y_N']^T$, todo el sistema cabe en una sola línea:

$$\mathbf{y}' = \mathbf{f}(x,\mathbf{y})$$

¡Para sistemas de EDOs de primer orden, la función de pendiente escalar $f(x,y)$ se reemplaza por el campo vectorial $\mathbf{f}(x,\mathbf{y})$!

Resolver el sistema de ecuaciones mediante el método de Euler es realmente sencillo: aproxime la derivada de cada variable dependiente en el sistema a través de diferencias finitas y resuelva para el valor de la función en el extremo derecho del intervalo sobre el que se aproxima. Básicamente, haga lo mismo que hicimos antes para deducir que $y_{k+1} = y_k + hf(x_k, y_k)$ pero *por cada función $y_i$ en el sistema de EDOs*.

De esta forma, por cada iteración $k$, $x_k = x_0 + hk$ y por cada variable dependiente:

$$
\begin{aligned}
(y_1)_{k+1} &= (y_1)_{k} + h f_1 (x_k, (y_1)_k, (y_2)_k, ..., (y_N)_k)\\
(y_2)_{k+1} &= (y_2)_{k} + h f_2 (x_k, (y_1)_k, (y_2)_k, ..., (y_N)_k)\\
&\ \ \vdots\\
(y_N)_{k+1} &= (y_N)_{k} + h f_N (x_k, (y_1)_k, (y_2)_k, ..., (y_N)_k)
\end{aligned}
$$

Estas $N$ fórmulas son exactamente las componentes de una sola ecuación vectorial. Por lo tanto, para un sistema de ecuaciones la fórmula iterativa del método de Euler puede reescribirse en forma vectorial:

$$\mathbf{y}_{k+1} =\mathbf{y}_{k} + h\,\mathbf{f}(x_k, \mathbf{y}_{k})$$

Esta forma de expresar la fórmula iterativa del método de Euler es extremadamente importante. Por favor, recuérdela, que será de mucha utilidad cuando tratemos de estudiar la estabilidad del método en este caso.

## **1.3. Sistemas de EDOs de orden superior**
Ahora considere que agregamos la complejidad adicional de que en nuestro sistema de ecuaciones tenemos al menos una ecuación con una derivada de orden superior de una variable dependiente. Suponga, por ejemplo, que en un sistema de ecuaciones que involucra las variables $\textbf{y} = [y_1, y_2, ..., y_N]^T$ la variable $y_i$, con $i \in \{1, 2, \dots, N\}$, tiene una derivada de orden $j$-ésimo en una ecuación del sistema.

La solución, en este caso, es bastante simple: *reemplace la derivada de orden superior por una derivada de primer orden utilizando variables auxiliares*.

Para ilustrar mejor esta idea, olvide que tratamos con sistemas de ecuaciones y considere lo que ocurre con una sola EDO de segundo orden:

$$\frac{d^2y}{dt^2} = \frac{dy}{dt} - 1$$

En este caso, podemos convertir la ecuación de segundo orden en un *sistema* de ecuaciones de primer orden sustituyendo la derivada de segundo orden por la derivada de una variable auxiliar. Considere que $u_1 = y$ y $u_2 = y'$. Para este problema se obtiene el sistema de EDOs de primer orden:

$$\begin{cases}
\frac{du_2}{dt}= u_2 - 1\\
\frac{du_1}{dt}= u_2
\end{cases}
$$

Por lo tanto, en un sistema de ecuaciones donde aparecen múltiples EDOs de orden superior, cada EDO de orden $n$ se descompone en exactamente $n$ EDOs de primer orden, usando las variables auxiliares $u_1 = y,\ u_2 = y',\ \dots,\ u_n = y^{(n-1)}$. En este ejemplo, dado que la ecuación era de segundo orden (orden 2), se necesitaron 2 ecuaciones de primer orden.

Observe que el modelo del motor ya está escrito como un sistema de primer orden, con vector de estado $\mathbf{y} = [i,\ \omega,\ \theta]^T$: la ecuación $\frac{d\theta}{dt} = \omega$ es justamente la reducción de orden de la ecuación de segundo orden para $\theta$.

Con esto mencionado, ya tenemos la teoría suficiente para (1) aplicar el método de Euler y generalizar a otros métodos para resolver EDOs vistos en el curso y (2) Analizar la estabilidad de la solución numérica, al menos para el método de Euler.

## **1.4. Estabilidad Numérica del Método de Euler**
Suponga que la solución exacta del sistema de EDOs en $x_k$ es $\mathbf{Y}(x_k)$ (observe que se usó una letra mayúscula). Esta puede calcularse de forma exacta en las cercanías de $x_k$ al utilizar series de Taylor y agregar un término de error local $\mathbf{T}_k$ por los términos de orden superior:

$$\mathbf{Y}(x_{k+1}) = \mathbf{Y}(x_k) + h\,\mathbf{f}(x_k, \mathbf{Y}(x_k)) + \mathbf{T}_k$$

Como $\mathbf{Y}$ es una función vectorial, el residuo de Taylor se escribe componente a componente: cada componente tiene su propio punto intermedio $\epsilon_i \in (x_k, x_{k+1})$,

$$(\mathbf{T}_k)_i = \frac{h^2}{2}\,Y_i''(\epsilon_i), \qquad i = 1, 2, \dots, N$$

En general no existe un único $\epsilon$ que sirva para todas las componentes a la vez. Lo que sí se puede afirmar es la cota

$$\|\mathbf{T}_k\| \le \frac{h^2}{2}\max_{x\in[x_k,x_{k+1}]}\|\mathbf{Y}''(x)\|$$

Definimos el error en el paso $k$ como $\mathbf{e}_{k} = \mathbf{Y}(x_{k}) - \mathbf{y}_{k}$. Al restar la fórmula de Euler $\mathbf{y}_{k+1} = \mathbf{y}_k + h\,\mathbf{f}(x_k, \mathbf{y}_k)$ de la expansión de Taylor, el error en el paso $k+1$ es:

$$\mathbf{e}_{k+1} = \mathbf{Y}(x_{k+1}) - \mathbf{y}_{k+1} = [\mathbf{Y}(x_k)-\mathbf{y}_{k}] + h[\mathbf{f}(x_k, \mathbf{Y}(x_k)) - \mathbf{f}(x_k, \mathbf{y}_k)] + \mathbf{T}_k$$

Para tratar la diferencia $\mathbf{f}(x_k, \mathbf{Y}(x_k)) - \mathbf{f}(x_k, \mathbf{y}_k)$ ahora sí necesitamos al Jacobiano. Recuerde cómo se define:

Sea $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m$ una función vectorial diferenciable definida por $\mathbf{f}(\mathbf{x}) = (f_1(\mathbf{x}), f_2(\mathbf{x}), \dots, f_m(\mathbf{x}))$, donde $\mathbf{x} = (x_1, x_2, \dots, x_n)$. La *matriz jacobiana* de $\mathbf{f}$ en $\mathbf{x}$, denotada por $\mathbf{J}_{\mathbf{f}}(\mathbf{x})$ o $\frac{\partial \mathbf{f}}{\partial \mathbf{x}}$, es la matriz de $m \times n$ de sus primeras derivadas parciales:

$$
\mathbf{J}_{\mathbf{f}}(\mathbf{x}) = \frac{\partial (f_1, \dots, f_m)}{\partial (x_1, \dots, x_n)} =
\begin{pmatrix}
\frac{\partial f_1}{\partial x_1} & \frac{\partial f_1}{\partial x_2} & \cdots & \frac{\partial f_1}{\partial x_n} \\
\frac{\partial f_2}{\partial x_1} & \frac{\partial f_2}{\partial x_2} & \cdots & \frac{\partial f_2}{\partial x_n} \\
\vdots & \vdots & \ddots & \vdots \\
\frac{\partial f_m}{\partial x_1} & \frac{\partial f_m}{\partial x_2} & \cdots & \frac{\partial f_m}{\partial x_n}
\end{pmatrix}
$$

En nuestro caso nos interesa el Jacobiano del campo vectorial $\mathbf{f}$ respecto de las variables dependientes $\mathbf{y}$, que es una matriz de $N \times N$:

$$
\mathbf{J}_{\mathbf{f}}(x,\mathbf{y}) = \frac{\partial \mathbf{f}}{\partial \mathbf{y}} =
\begin{pmatrix}
\frac{\partial f_1}{\partial y_1} & \cdots & \frac{\partial f_1}{\partial y_N} \\
\vdots & \ddots & \vdots \\
\frac{\partial f_N}{\partial y_1} & \cdots & \frac{\partial f_N}{\partial y_N}
\end{pmatrix}
$$

Por el teorema del valor medio (en su forma integral), si $\mathbf{f}$ es diferenciable respecto de $\mathbf{y}$:

$$\mathbf{f}(x_k,\mathbf{Y}(x_k)) - \mathbf{f}(x_k,\mathbf{y}_k) = \left[\int_0^1 \mathbf{J}_{\mathbf{f}}\big(x_k,\ \mathbf{y}_k + s\,\mathbf{e}_k\big)\,ds\right]\mathbf{e}_k \approx \mathbf{J}_k\,\mathbf{e}_k$$

donde $\mathbf{J}_k := \mathbf{J}_{\mathbf{f}}(x_k,\mathbf{y}_k)$ es el Jacobiano evaluado en el paso $k$. La aproximación es exacta si $\mathbf{f}$ es lineal en $\mathbf{y}$; en general, el error que se comete es $O(\|\mathbf{e}_k\|^2)$.

Por lo tanto, el error en el paso $k+1$ se vuelve

$$\mathbf{e}_{k+1} \approx \underbrace{(\mathbf{I} + h\mathbf{J}_k)\,\mathbf{e}_k}_{\text{propagación del error}} + \underbrace{\mathbf{T}_k}_{\text{error local nuevo},\ O(h^2)}$$

donde $\mathbf{I}$ es la matriz identidad de $N \times N$.

La recurrencia tiene dos partes. El término $\mathbf{T}_k$ es el error de truncamiento local que se inyecta en cada paso. El análisis de **estabilidad** se concentra en la primera parte: cómo se amplifica o se atenúa un error que ya existe, que es lo que produce la divergencia a gran escala. Por eso, de ahora en adelante estudiamos la recurrencia homogénea:

$$\mathbf{e}_{k+1} \approx (\mathbf{I} + h\mathbf{J}_k)\,\mathbf{e}_{k}$$

Observe que el error en el paso $k+1$ es proporcional al error en el paso anterior $k$ a través de una **matriz de amplificación** $(\mathbf{I}+ h\mathbf{J}_k)$. Pero, por la manera en que aquí está definida, esta matriz es en realidad un operador que cambia en cada paso: en un sistema no lineal, $\mathbf{J}_k$ depende de $\mathbf{y}_k$. A lo largo de $n$ pasos,

$$\mathbf{e}_n \approx (\mathbf{I}+h\mathbf{J}_{n-1})\cdots(\mathbf{I}+h\mathbf{J}_1)(\mathbf{I}+h\mathbf{J}_0)\,\mathbf{e}_0$$

que es un producto de matrices distintas, no una potencia de una sola matriz.

### **Observación sobre la fricción de Coulomb**
El análisis anterior supone que $\mathbf{f}$ es diferenciable respecto de $\mathbf{y}$. En el modelo del motor esto no se cumple en todas partes: $\frac{\partial}{\partial\omega}\text{sign}(\omega) = 0$ para $\omega \neq 0$ y no existe en $\omega = 0$. Por lo tanto:

1. Para $\omega \neq 0$, el término $\tau_c\,\text{sign}(\omega)$ es la constante $\pm\tau_c$, y el modelo es afín por tramos. Alrededor de un punto de operación con $\omega \neq 0$ el sistema sí puede linealizarse; la única no linealidad es el salto en $\omega = 0$, y ahí el sistema no se puede linealizar.
2. El análisis de estabilidad con $\mathbf{J}_{\mathbf{f}}$ vale por tramos ($\omega > 0$ y $\omega < 0$).
3. Para $\omega \neq 0$, con vector de estado $\mathbf{y} = [i,\ \omega,\ \theta]^T$, el Jacobiano del motor es
$$
\mathbf{J}_{\mathbf{f}} =
\begin{pmatrix}
-\dfrac{R}{L} & -\dfrac{K_e}{L} & 0 \\[6pt]
\dfrac{K_t}{J_m} & -\dfrac{b}{J_m} & 0 \\[6pt]
0 & 1 & 0
\end{pmatrix}
$$
4. Cuando el motor debería quedarse detenido ($\omega = 0$ con $|K_t i - \tau_{carga}| < \tau_c$), el método de Euler hace que $\omega$ oscile alrededor de cero (*chattering*) con una amplitud del orden de $h\tau_c/J_m$. Una alternativa habitual es regularizar con $\text{sign}(\omega) \approx \tanh(\omega/\varepsilon)$.